In [35]:
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from pathlib import Path
import os
os.environ["TRANSFORMERS_NO_TRAINER"] = "1"


In [2]:

RESULTS_DIR = Path("/home/vivekbisht/Desktop/faers-signal-detection/results")


In [3]:
for table_name in ['demo', 'drug', 'reac']:
    file_path = RESULTS_DIR / f"{table_name}.parquet"
    size_mb = file_path.stat().st_size / 1024 / 1024
    pq_file = pq.ParquetFile(file_path)
    nr =pq_file.metadata.num_rows
    sam = pd.read_parquet(file_path,engine='pyarrow').head(1000)
    print(f"{table_name}->[size={size_mb}mb,num_rows={nr},sample_row=\n{sam}]") 

demo->[size=215.4486789703369mb,num_rows=16645788,sample_row=
     PRIMARYID    CASEID   AGE AGE_COD  SEX             quarter
0    100035813  10003581   NaN     NaN  NaN  faers_ascii_2018q1
1    100048965  10004896  64.0      YR    F  faers_ascii_2018q1
2    100054258  10005425  71.0      YR    M  faers_ascii_2018q1
3    100058884  10005888  54.0      YR    M  faers_ascii_2018q1
4    100060834  10006083   NaN     NaN    F  faers_ascii_2018q1
..         ...       ...   ...     ...  ...                 ...
995  107655494  10765549  70.0      YR    M  faers_ascii_2018q1
996  107660932  10766093   NaN     NaN    F  faers_ascii_2018q1
997  107670305  10767030  50.0      YR    M  faers_ascii_2018q1
998  107671782  10767178   NaN     NaN    F  faers_ascii_2018q1
999  107675122  10767512  35.0      YR    F  faers_ascii_2018q1

[1000 rows x 6 columns]]
drug->[size=229.35858917236328mb,num_rows=17022628,sample_row=
     PRIMARYID    CASEID ROLE_COD       DRUGNAME             quarter
0    1000358

In [4]:
#Drug -Event Pair   

In [5]:
chksize = 1_000_000

In [6]:
# def drug_event_pair():
#     demo_reader = pd.read_parquet(RESULTS_DIR / 'demo.parquet',columns=['CASEID',"PRIMARYID",'quarter']  , engine='pyarrow')
#     demo_clean = demo_reader[["PRIMARYID",'CASEID','quarter']].drop_duplicates()
#     demo_clean = demo_clean.set_index("PRIMARYID")
    
#     drug_reader = pd.read_parquet(RESULTS_DIR / 'drug.parquet',columns=["PRIMARYID",'DRUGNAME'], engine='pyarrow')
#     reac_reader = pd.read_parquet(RESULTS_DIR / 'reac.parquet',columns=["PRIMARYID",'PT'],engine='pyarrow')

#     drug_reac = drug_reader.merge(reac_reader, on='PRIMARYID', how='inner')

#     drug_reac = drug_reac.join(demo_clean, on='PRIMARYID', how='inner')

#     pairs = drug_reac[['CASEID', 'quarter', 'DRUGNAME', 'PT']].copy()
#     pairs.columns = ['caseid', 'quarter', 'drug', 'event']
#     pairs = pairs.drop_duplicates()
#     pairs['drug'] = pairs['drug'].str.upper().str.strip()
#     pairs['event'] = pairs['event'].str.upper().str.strip()
#     pairs = pairs.dropna()

#     output_path = RESULTS_DIR / 'drug_event_pairs.parquet'
#     pairs.to_parquet(output_path , engine='pyarrow' ,index=False)
#     return pairs


In [7]:
# pairs = drug_event_pair()  facing exploding ram issues movign to chunking

In [ ]:


def drug_event_pairs():
   
    demo = pd.read_parquet(RESULTS_DIR / 'demo.parquet',columns=['CASEID', 'PRIMARYID', 'quarter'],engine='pyarrow')   
    demo_clean = demo[['PRIMARYID', 'CASEID', 'quarter']].drop_duplicates()
  
    duplicated_pids = demo_clean[demo_clean['PRIMARYID'].duplicated(keep=False)]
    if len(duplicated_pids) > 0:
        demo_clean = demo_clean.drop_duplicates(subset=['PRIMARYID'], keep='first')
     
    

    demo_dict = demo_clean.set_index('PRIMARYID')[['CASEID', 'quarter']].to_dict('index')

    del demo, demo_clean
    
  
    drug_file = pq.ParquetFile(RESULTS_DIR / 'drug.parquet')
    reac_file = pq.ParquetFile(RESULTS_DIR / 'reac.parquet')
    
    output_path = RESULTS_DIR / 'drug_event_pairs.parquet'
    writer = None
    total_rows = 0
    
    batch_size = 500_000
    

    for drug_batch_num, drug_batch in enumerate(
        drug_file.iter_batches(batch_size=batch_size, columns=['PRIMARYID', 'DRUGNAME']),
        start=1
    ):
        drug_chunk = drug_batch.to_pandas()
        drug_chunk['DRUGNAME'] = drug_chunk['DRUGNAME'].str.upper().str.strip()
     
    
        drug_pids = set(drug_chunk['PRIMARYID'].values)
       
        
        batch_pairs = []
        
        for reac_batch_num, reac_batch in enumerate(reac_file.iter_batches(batch_size=batch_size, columns=['PRIMARYID', 'PT']),start=1):

            reac_chunk = reac_batch.to_pandas()
            reac_chunk['PT'] = reac_chunk['PT'].str.upper().str.strip()
            
            reac_filtered = reac_chunk[reac_chunk['PRIMARYID'].isin(drug_pids)]
            
            if len(reac_filtered) == 0:
                continue
            
            
            pairs = drug_chunk.merge(reac_filtered, on='PRIMARYID', how='inner')
            
            if len(pairs) > 0:
                batch_pairs.append(pairs)
        

        if not batch_pairs:
           
            continue
        
        drd = pd.concat(batch_pairs, ignore_index=True)
    
        drd['CASEID'] = drd['PRIMARYID'].map(lambda x: demo_dict.get(x, {}).get('CASEID'))
        drd['quarter'] = drd['PRIMARYID'].map(lambda x: demo_dict.get(x, {}).get('quarter'))
    
        drd = drd[['CASEID', 'quarter', 'DRUGNAME', 'PT']].copy()
        drd.columns = ['caseid', 'quarter', 'drug', 'event']
        
       
        drd = drd.dropna().drop_duplicates()
       
        
        if drd.empty:
            continue
        
        
        table = pa.Table.from_pandas(drd, preserve_index=False)
        
        if writer is None:
            writer = pq.ParquetWriter(output_path, table.schema)

        
        writer.write_table(table)

       
    
    if writer:
        writer.close()

    else:
        print("\n No data written")
    
    return drd



In [15]:
drd = drug_event_pairs()